**Цель данного ноутбука:** очистить данные в основной таблице с заказами и подготовить удобную ветрину данных для дальнейшего анализа

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import pandas as pd
import numpy as np

In [17]:
PATH_TO_DATA = '/content/drive/MyDrive/Datasets/olist-ecommerce-analytics/data/'

In [28]:
df_orders = pd.read_csv(PATH_TO_DATA + 'olist_orders_dataset.csv')
df_items = pd.read_csv(PATH_TO_DATA + 'olist_order_items_dataset.csv')
df_products = pd.read_csv(PATH_TO_DATA + 'olist_products_dataset.csv')
df_translation = pd.read_csv(PATH_TO_DATA + 'product_category_name_translation.csv')
df_customers = pd.read_csv(PATH_TO_DATA + 'olist_customers_dataset.csv')
df_payments = pd.read_csv(PATH_TO_DATA + 'olist_order_payments_dataset.csv')
df_geolocation = pd.read_csv(PATH_TO_DATA + 'olist_geolocation_dataset.csv')
df_sellers = pd.read_csv(PATH_TO_DATA + 'olist_sellers_dataset.csv')
df_review = pd.read_csv(PATH_TO_DATA + 'olist_order_reviews_dataset.csv')

1. Подготовка и очистка данных в таблице `df_orders`



In [29]:
# Распределение статусов

print("*** СТАТУСЫ ЗАКАЗОВ ***")
print('\n')
print(df_orders['order_status'].value_counts())

*** СТАТУСЫ ЗАКАЗОВ ***


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [30]:
# Проверка пропусков в таблице заказов

print("*** ПРОПУСКИ В ТАБЛИЦЕ ЗАКАЗОВ (df_orders) ***")
print('\n')
print(df_orders.isnull().sum())

*** ПРОПУСКИ В ТАБЛИЦЕ ЗАКАЗОВ (df_orders) ***


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [31]:
# Проверка уникальности ключей

print("*** ПРОВЕРКА НА ДУБЛИКАТЫ ***")
print('\n')
print(f"Всего строк в df_orders: {len(df_orders)}")
print(f"Уникальных order_id в df_orders: {df_orders['order_id'].nunique()}")

*** ПРОВЕРКА НА ДУБЛИКАТЫ ***


Всего строк в df_orders: 99441
Уникальных order_id в df_orders: 99441


In [35]:
# Поскольку, мне интересно в дальнейшем рассматривать только завершенные заказы со статусом 'delivered', то я фильтрую талицу по полю 'order_status'

df_orders_delivered = df_orders[df_orders['order_status'] == 'delivered']

In [36]:
# Проверка на пропуски уже в отфильрованной таблице со статуром 'delivered'

print("*** Пропуски в отфильтованной таблице (df_orders_delivered) ***")
print('\n')
print(df_orders_delivered.isnull().sum())

*** Пропуски в отфильтованной таблице (df_orders_delivered) ***


order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [37]:
# Оставляею только доставленные И только те, у которых дата доставки НЕ пустая

df_orders_clean = df_orders_delivered.dropna(subset=['order_delivered_customer_date'])

In [38]:
# Оставляею только доставленные И только те, у которых дата доставки НЕ пустая

df_orders_clean = df_orders_delivered.dropna(subset=['order_delivered_customer_date'])

## Проверка качества данных:

1. Дубликаты по `order_id` не обноружены
2. В ходе анализа отфильтрованных данных по полю `order_status` были обнаружены аномалии: **8 заказов** успешно доставлены по документам, но не имеют физической даты вручения клиенту

**Возможные причины:**
> 1. *Человеческий фактор:* Курьер фактически передал посылку, но забыл закрыть заявку в мобильном приложении.
> 2. *Технический сбой:* Ошибка при синхронизации баз данных
>
**Решение:** Данные 8 строк были удалены из выборки. Оставление их в датасете может привести при последующем расчете операционной эффективности к ошибкам в коде или исказят средние показатели работы логистики.

2. Объединение данных

In [58]:
df_sales_full = df_orders_clean.merge(
    df_items,
    on='order_id',
    how='left'
).merge(
    df_customers,
    on='customer_id',
    how='left'
).merge(
    df_products,
    on='product_id',
    how='left'
).merge(
    df_payments,
    on='order_id',
    how='left'
)

In [48]:
# Создаю словарь для перевода категорий товара с португальского на английский

category_dict = df_translation.set_index('product_category_name')['product_category_name_english'].to_dict()

In [59]:
# Заменяю значения из словаря в основной таблице

df_sales_full['product_category_name'] = df_sales_full['product_category_name'].map(category_dict)

In [74]:
# Проверяю колнки и типы данных

df_sales_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115030 entries, 0 to 115029
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   order_id                  115030 non-null  object        
 1   customer_id               115030 non-null  object        
 2   order_status              115030 non-null  object        
 3   order_purchase_timestamp  115030 non-null  datetime64[ns]
 4   order_item_id             115030 non-null  int64         
 5   product_id                115030 non-null  object        
 6   seller_id                 115030 non-null  object        
 7   price                     115030 non-null  float64       
 8   freight_value             115030 non-null  float64       
 9   customer_unique_id        115030 non-null  object        
 10  customer_city             115030 non-null  object        
 11  customer_state            115030 non-null  object        
 12  pr

In [64]:
# Замечаю, что колонки с датами имеют тип object, перевожу их в формат datetime, для удобства в дальнейших расчетах

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    df_sales_full[col] = pd.to_datetime(df_sales_full[col])

In [66]:
df_sales_full[date_columns].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [67]:
# Проверяю на дубликаты после всех объеденений таблиц

df_sales_full.duplicated().sum()

np.int64(0)

In [69]:
df_sales_full.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,268.0,4.0,500.0,19.0,8.0,13.0,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,268.0,4.0,500.0,19.0,8.0,13.0,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,268.0,4.0,500.0,19.0,8.0,13.0,2.0,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,...,178.0,1.0,400.0,19.0,13.0,19.0,1.0,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,...,232.0,1.0,420.0,24.0,19.0,21.0,1.0,credit_card,3.0,179.12


In [72]:
# Получается грамоздкая таблица с 30 кологками, дропаю мусорные колонки

columns_to_drop = [
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'shipping_limit_date', 'payment_sequential', 'customer_zip_code_prefix',
    'order_approved_at', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date'
]

df_sales_full = df_sales_full.drop(columns=columns_to_drop, errors='ignore')

In [73]:
df_sales_full.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_item_id,product_id,seller_id,price,freight_value,customer_unique_id,customer_city,customer_state,product_category_name,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,housewares,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,housewares,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,housewares,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,af07308b275d755c9edb36a90c618231,barreiras,BA,perfumery,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,auto,credit_card,3.0,179.12


In [75]:
# Сохраняю финальную чистую витрину на Google Диск

df_sales_full.to_csv(PATH_TO_DATA + 'olist_clean_sales.csv', index=False)

In [78]:
# Финальная структура файла

df_sales_full.shape

(115030, 16)